# Session 5 · Build and harden a lake

Catawba Precision Components, a fictional Charlotte-area manufacturer, builds the governed data lake its plant telemetry lands in.

In [ ]:
source ~/dsba6190-live-demo-05/env.sh && echo "lake: dsba6190-lake-$SUFFIX" \
  || echo "NOT STAGED. Run prep.ipynb first. Do not run any other cell."

## Step 1 · Read the governed bucket before anything runs

In [ ]:
cd "$WORK"

cp lake.tf.staged lake.tf

cat lake.tf


terraform plan

## Step 2 · Apply. Two buckets, one governed

In [ ]:
cd "$WORK"

terraform apply $AUTO_APPROVE

gcloud storage buckets describe "gs://dsba6190-lake-$SUFFIX" \
  --project "$PROJECT" \
  --format="yaml(name,location,default_storage_class,versioning_enabled,uniform_bucket_level_access,public_access_prevention,labels)"

gcloud storage buckets describe "gs://dsba6190-staging-$SUFFIX" \
  --project "$PROJECT" \
  --format="yaml(name,location,default_storage_class,versioning_enabled,uniform_bucket_level_access,public_access_prevention,labels)"

## Step 3 · The four zones, and the object that lands in raw

In [ ]:
cd "$WORK"

In [ ]:
# Creating 5 zone sentinel objects (raw, validated, curated, archive, quarantine)...
for z in raw validated curated archive quarantine; do
  printf 'zone: %s\ncontract: see lecture 5, slide 12\n' "$z" > _zone.txt
  gcloud storage cp _zone.txt "gs://dsba6190-lake-$SUFFIX/$z/_ZONE" --project "$PROJECT"
done

gcloud storage ls "gs://dsba6190-lake-$SUFFIX" --project "$PROJECT"

In [ ]:
# Uploading sample/readings.csv to gs://dsba6190-lake-$SUFFIX/raw/readings/readings.csv...
gcloud storage cp sample/readings.csv "gs://dsba6190-lake-$SUFFIX/raw/readings/readings.csv" --project "$PROJECT"

gcloud storage ls -r "gs://dsba6190-lake-$SUFFIX/raw/" --project "$PROJECT"

## Step 4 · Make it public. Twice

In [ ]:
cd "$WORK"

In [ ]:
# 1. Adding allUsers objectViewer binding to ungoverned staging bucket...
gcloud storage buckets add-iam-policy-binding "gs://dsba6190-staging-$SUFFIX" \
  --project "$PROJECT" \
  --member=allUsers --role=roles/storage.objectViewer

In [ ]:
# 2. Uploading confidential defect report _qc.csv to staging bucket...
printf 'plant,serial,defect\ncharlotte,SN-88213,hairline crack\n' > _qc.csv
gcloud storage cp _qc.csv "gs://dsba6190-staging-$SUFFIX/qc.csv" --project "$PROJECT"

for i in $(seq 1 12); do
  body="$(curl -s "https://storage.googleapis.com/dsba6190-staging-$SUFFIX/qc.csv" || true)"
  if [[ "$body" == *"hairline"* ]]; then
    echo "$body"
    echo -e "\033[1;31m[DANGER]\033[0m Anonymous internet fetch succeeded without credentials!"
    break
  fi
  echo "Waiting for IAM propagation ($i/12)..."
  sleep 2
done

In [ ]:
# 3. Attempting the SAME allUsers binding on the GOVERNED lake bucket...
gcloud storage buckets add-iam-policy-binding "gs://dsba6190-lake-$SUFFIX" \
  --project "$PROJECT" \
  --member=allUsers --role=roles/storage.objectViewer 2>&1
PAP_CODE=$?

In [ ]:
# 4. Attempting per-object ACL grant on the GOVERNED lake bucket...
gcloud storage objects update "gs://dsba6190-lake-$SUFFIX/raw/readings/readings.csv" \
  --project "$PROJECT" \
  --add-acl-grant=entity=allUsers,role=READER 2>&1
UBLA_CODE=$?

In [ ]:
# 5. Revoking public exposure on the staging bucket...
gcloud storage buckets remove-iam-policy-binding "gs://dsba6190-staging-$SUFFIX" \
  --project "$PROJECT" \
  --member=allUsers --role=roles/storage.objectViewer

## Step 5 · Versioning is the undo

In [ ]:
cd "$WORK"

In [ ]:
# 1. Creating and uploading manifest v1 (200,000 rows)...
printf '{"rows": 200000, "written_by": "nightly-ingest", "status": "complete"}\n' > _manifest.json
gcloud storage cp _manifest.json "gs://dsba6190-lake-$SUFFIX/raw/readings/_manifest.json" --project "$PROJECT"

GEN_GOOD="$(gcloud storage objects describe "gs://dsba6190-lake-$SUFFIX/raw/readings/_manifest.json" \
  --project "$PROJECT" --format='value(generation)')"

In [ ]:
# 2. Corrupting/overwriting with rerun nobody reviewed (0 rows)...
printf '{"rows": 0, "written_by": "the-rerun-nobody-reviewed", "status": "complete"}\n' > _manifest.json
gcloud storage cp _manifest.json "gs://dsba6190-lake-$SUFFIX/raw/readings/_manifest.json" --project "$PROJECT"

gcloud storage cat "gs://dsba6190-lake-$SUFFIX/raw/readings/_manifest.json" --project "$PROJECT"

gcloud storage ls --all-versions --long "gs://dsba6190-lake-$SUFFIX/raw/readings/" --project "$PROJECT"

In [ ]:
# 3. Restoring prior generation #$GEN_GOOD...
gcloud storage cp "gs://dsba6190-lake-$SUFFIX/raw/readings/_manifest.json#$GEN_GOOD" \
  "gs://dsba6190-lake-$SUFFIX/raw/readings/_manifest.json" --project "$PROJECT"

gcloud storage cat "gs://dsba6190-lake-$SUFFIX/raw/readings/_manifest.json" --project "$PROJECT"

In [ ]:
# 4. Testing accidental DELETE...
gcloud storage rm "gs://dsba6190-lake-$SUFFIX/raw/readings/_manifest.json" --project "$PROJECT"

gcloud storage ls "gs://dsba6190-lake-$SUFFIX/raw/readings/" --project "$PROJECT"

gcloud storage ls --all-versions --long "gs://dsba6190-lake-$SUFFIX/raw/readings/_manifest.json" --project "$PROJECT"

In [ ]:
# 5. Restoring after delete from #$GEN_GOOD...
gcloud storage cp "gs://dsba6190-lake-$SUFFIX/raw/readings/_manifest.json#$GEN_GOOD" \
  "gs://dsba6190-lake-$SUFFIX/raw/readings/_manifest.json" --project "$PROJECT"
gcloud storage cat "gs://dsba6190-lake-$SUFFIX/raw/readings/_manifest.json" --project "$PROJECT"

## Step 6 · CSV to Parquet. Two arguments, two numbers

In [ ]:
cd "$WORK"

In [ ]:
# 1. Uploading sample/readings.parquet to Curated zone...
gcloud storage cp sample/readings.parquet \
  "gs://dsba6190-lake-$SUFFIX/curated/readings/readings.parquet" --project "$PROJECT"

gcloud storage ls -l \
  "gs://dsba6190-lake-$SUFFIX/raw/readings/readings.csv" \
  "gs://dsba6190-lake-$SUFFIX/curated/readings/readings.parquet" --project "$PROJECT"

In [ ]:
# 2. Staging and applying BigQuery external tables (tables.tf)...
cp tables.tf.staged tables.tf
terraform apply $AUTO_APPROVE

JOB_CSV="dsba6190-$SUFFIX-csv-$(date +%s)"
JOB_PARQ="dsba6190-$SUFFIX-parq-$(date +%s)"

In [ ]:
# 3. Querying CSV external table (Job: $JOB_CSV)...
bq --project_id="$PROJECT" query --use_legacy_sql=false --nouse_cache --job_id="$JOB_CSV" \
  "SELECT ROUND(AVG(value), 3) AS mean_reading FROM \`$PROJECT.dsba6190_lake_$SUFFIX.readings_csv\`"

In [ ]:
# 4. Querying Parquet external table (Job: $JOB_PARQ)...
bq --project_id="$PROJECT" query --use_legacy_sql=false --nouse_cache --job_id="$JOB_PARQ" \
  "SELECT ROUND(AVG(value), 3) AS mean_reading FROM \`$PROJECT.dsba6190_lake_$SUFFIX.readings_parquet\`"

bq --project_id="$PROJECT" show -j "$JOB_CSV"

bq --project_id="$PROJECT" show -j "$JOB_PARQ"

## Step 7 · The lifecycle rule that nothing runs tonight

In [ ]:
cd "$WORK"

cp lake.tf.lifecycle lake.tf

terraform plan

terraform apply $AUTO_APPROVE

gcloud storage buckets describe "gs://dsba6190-lake-$SUFFIX" \
  --project "$PROJECT" \
  --format="yaml(name,lifecycle_config)"

## Step 8 · Retention, and a delete that fails

In [ ]:
cd "$WORK"

In [ ]:
# 1. Staging and applying vault bucket with retention policy...
cp vault.tf.staged vault.tf
terraform apply $AUTO_APPROVE

gcloud storage buckets describe "gs://dsba6190-vault-$SUFFIX" \
  --project "$PROJECT" \
  --format="yaml(name,retention_policy)"

In [ ]:
# 2. Writing and uploading critical incident record _incident.csv...
printf 'incident,plant,reading,recorded_at\n1,charlotte,231.4,2026-09-17T19:04:11Z\n' > _incident.csv
gcloud storage cp _incident.csv "gs://dsba6190-vault-$SUFFIX/raw/incident.csv" --project "$PROJECT"

In [ ]:
# 3. Attempting to delete retention-protected incident record...
gcloud storage rm "gs://dsba6190-vault-$SUFFIX/raw/incident.csv" --project "$PROJECT" 2>&1
DELETE_CODE=$?

## Step 9 · CMEK, and crypto-shredding

In [ ]:
KEYRING="dsba6190-demo"
KEYNAME="lake-cmek"
REGION="us-east1"



cd "$WORK"

In [ ]:
# 1. Staging and applying CMEK-encrypted secure bucket...
cp secure.tf.staged secure.tf
terraform apply $AUTO_APPROVE

In [ ]:
# 2. Writing and uploading regulated data (_regulated.csv)...
printf 'patient_site,serial,defect\ncharlotte,SN-88213,hairline crack\n' > _regulated.csv
gcloud storage cp _regulated.csv "gs://dsba6190-secure-$SUFFIX/raw/regulated.csv" --project "$PROJECT"

gcloud storage objects describe "gs://dsba6190-secure-$SUFFIX/raw/regulated.csv" \
  --project "$PROJECT" --format="yaml(name,kms_key)"

gcloud storage cat "gs://dsba6190-secure-$SUFFIX/raw/regulated.csv" --project "$PROJECT"

In [ ]:
# 3. Disabling Cloud KMS key version (Crypto-shredding)...
VERSION="$(gcloud kms keys versions list --key "$KEYNAME" --keyring "$KEYRING" \
  --location "$REGION" --project "$PROJECT" --filter="state=ENABLED" \
  --format="value(name)" --limit=1 | awk -F/ '{print $NF}')"
VERSION="${VERSION:-1}"

gcloud kms keys versions disable "$VERSION" --key "$KEYNAME" --keyring "$KEYRING" \
  --location "$REGION" --project "$PROJECT"

In [ ]:
# 4. Testing READ with disabled key...
for i in $(seq 1 6); do
  body="$(gcloud storage cat "gs://dsba6190-secure-$SUFFIX/raw/regulated.csv" --project "$PROJECT" 2>&1)"
  status=$?
  if [[ "$body" == *"KEY_DISABLED"* ]] || [[ "$body" == *"ERROR"* ]]; then
    echo "$body"
    break
  else
    echo "Cached unwrapped key in transit; retrying in 2s ($i/6)..."
    sleep 2
  fi
done

In [ ]:
# 5. Testing WRITE with disabled key (refuses immediately)...
gcloud storage cp _regulated.csv "gs://dsba6190-secure-$SUFFIX/raw/second.csv" --project "$PROJECT" 2>&1

In [ ]:
# 6. Re-enabling Cloud KMS key version $VERSION...
gcloud kms keys versions enable "$VERSION" --key "$KEYNAME" --keyring "$KEYRING" \
  --location "$REGION" --project "$PROJECT"

for i in $(seq 1 6); do
  body="$(gcloud storage cat "gs://dsba6190-secure-$SUFFIX/raw/regulated.csv" --project "$PROJECT" 2>&1)"
  status=$?
  if [ $status -eq 0 ]; then
    echo "$body"
    break
  fi
  sleep 2
done

## Step 10 · Partition pruning, measured

In [ ]:
cd "$WORK"

In [ ]:
# 1. Uploading partitioned events to Curated zone...
gcloud storage cp -r sample/events "gs://dsba6190-lake-$SUFFIX/curated/" --project "$PROJECT"

gcloud storage ls "gs://dsba6190-lake-$SUFFIX/curated/events/" --project "$PROJECT"

In [ ]:
# 2. Staging and applying Hive-partitioned external table (events.tf)...
cp events.tf.staged events.tf
cat events.tf
terraform apply $AUTO_APPROVE

JOB_ALL="dsba6190-$SUFFIX-all-$(date +%s)"
JOB_ONE="dsba6190-$SUFFIX-one-$(date +%s)"

In [ ]:
# 3. Querying ALL partitions (Job: $JOB_ALL)...
bq --project_id="$PROJECT" query --use_legacy_sql=false --nouse_cache --job_id="$JOB_ALL" \
  "SELECT COUNT(*) AS readings, ROUND(AVG(value), 3) AS mean_reading FROM \`$PROJECT.dsba6190_lake_$SUFFIX.events\`"

In [ ]:
# 4. Querying ONE partition with WHERE dt = '2026-09-17' (Job: $JOB_ONE)...
bq --project_id="$PROJECT" query --use_legacy_sql=false --nouse_cache --job_id="$JOB_ONE" \
  "SELECT COUNT(*) AS readings, ROUND(AVG(value), 3) AS mean_reading FROM \`$PROJECT.dsba6190_lake_$SUFFIX.events\` WHERE dt = '2026-09-17'"

bq --project_id="$PROJECT" show -j "$JOB_ALL"

bq --project_id="$PROJECT" show -j "$JOB_ONE"

## Step 11 · The guardrail above the project

In [ ]:
cd "$WORK"

gcloud projects get-ancestors "$PROJECT"

gcloud organizations list || true

gcloud org-policies list --project="$PROJECT" || true

gcloud org-policies describe constraints/storage.publicAccessPrevention \
  --project="$PROJECT" --effective || true

In [ ]:
# Attempting to set project-level organization policy (_pap.yaml)...
cat > _pap.yaml <<YAML
name: projects/$PROJECT/policies/storage.publicAccessPrevention
spec:
  rules:
  - enforce: true
YAML

gcloud org-policies set-policy _pap.yaml --project="$PROJECT" 2>&1

## Step 12 · Teardown is a teaching step

In [ ]:
KEYRING="dsba6190-demo"
KEYNAME="lake-cmek"
REGION="us-east1"



cd "$WORK"

In [ ]:
# 1. Attempting terraform destroy (Watch for expected retention refusal!)...
terraform destroy $AUTO_APPROVE
DESTROY_STATUS=$?

if [ $DESTROY_STATUS -ne 0 ]; then
  echo -e "\n\033[1;33m[Teaching note]\033[0m Destruction failed on the vault bucket as intended!"
  echo "The retention policy protects data even from Terraform destroy."
  echo "Non-atomic destruction: 7 resources were destroyed, 1 survived."
fi

In [ ]:
# 2. Clearing retention period on vault bucket...
gcloud storage buckets update "gs://dsba6190-vault-$SUFFIX" \
  --project "$PROJECT" \
  --clear-retention-period || true

In [ ]:
# 3. Re-running terraform destroy (Now succeeds)...
terraform destroy $AUTO_APPROVE

In [ ]:
# 4. Scheduling Cloud KMS key version destruction (Optional cleanup)...
VERSION="$(gcloud kms keys versions list --key "$KEYNAME" --keyring "$KEYRING" \
  --location "$REGION" --project "$PROJECT" --filter="state=ENABLED" \
  --format="value(name)" --limit=1 | awk -F/ '{print $NF}' || true)"
if [ -n "$VERSION" ]; then
  echo "Scheduling KMS key version $VERSION for destruction..."
  gcloud kms keys versions destroy "$VERSION" --key "$KEYNAME" --keyring "$KEYRING" \
    --location "$REGION" --project "$PROJECT" || true
fi

In [ ]:
# 5. Final Estate Verification:
gcloud storage ls --project "$PROJECT" || echo "  No storage buckets remaining."

bq --project_id="$PROJECT" ls || echo "  No BigQuery datasets remaining."